# Analysing a joint dataset

Step 3 of the ISO pipeline: an end-to-end joint analysis of the primary CMB (MFLike TT/TE/EE) and the
CMB-lensing reconstruction, fitting **exactly the products built upstream**:

1. build the joint likelihood on the smooth datasets from
   [`create_datasets.ipynb`](create_datasets.ipynb) and evaluate it at the fiducial point;
2. fold in the **cross-covariance** from
   [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb);
3. launch an MCMC.

The configuration is an ordinary Cobaya `info` dict built from `defaults/`, so the dataset, accuracy
and cosmology stay explicit; `resolve_aliases` recovers the role aliases (`.mflike`, `.lensing`).
Because both smooth twins were imprinted at the same ISO fiducial we fit at, the joint chi-square is
**0 by construction**.

> **Configuration.** Every fiducial value, prior, modelling choice and file path comes from
> [`defaults/`](defaults/README.md), read through `iso_config.py`.
> `iso_config.build_info(dataset, model)` builds the `info` exactly as the matching `cobaya-grid`
> job does (see [`generate_grid.ipynb`](generate_grid.ipynb)), so what is evaluated here is what the
> grid runs. By default (`data="sims"`) the likelihoods read the smooth twins in `sims/`.

## 1. A joint likelihood on the smooth datasets

`iso_config.build_info("multigaussian_mflike_lensing")` returns the `info` of the grid dataset of the
same name: a single `MultiGaussianLikelihood` wiring MFLike + CMB lensing onto one shared CAMB theory.
`defaults/probes/*/sims.yaml` point each component at its smooth twin in `sims/` (the lensing
`correction`/`fiducial` aux files are referenced by **absolute** path from the shipped install, so we
never copy them). `iso_info` below fixes every parameter to the fiducial except those in `sample`,
and leaves out the cross-covariance unless asked, so the build points below stay identical.

In [ ]:
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

import iso_config
from soliket.presets import resolve_aliases

DATASET = "multigaussian_mflike_lensing"
XCOV = iso_config.path("cross_cov")


def loglike(model):
    """Total log-likelihood of a model at its fiducial point."""
    return float(sum(model.loglikes({})[0]))


def iso_info(sample=(), cross_cov=False):
    """Joint MFLike + lensing info from defaults/, wired to fit the sims/ twins.

    Every parameter is fixed to its fiducial value except those in `sample`;
    the cross-covariance is left out unless `cross_cov` is True.
    """
    info = iso_config.build_info(DATASET)
    info["params"] = iso_config.fix_to_fiducial(info["params"], sample=sample)
    if not cross_cov:
        del info["likelihood"][iso_config.MULTIGAUSSIAN]["cross_cov_path"]
    info["packages_path"] = resolve_packages_path()
    return info


info = iso_info()
model = get_model(info)
roles = resolve_aliases(model)

print("members :", type(roles.mflike).__name__, "+", type(roles.lensing).__name__)
print("joint loglike at fiducial:", round(loglike(model), 4), "(chi^2 = 0 twins)")

## 2. Adding the cross-covariance

Both probes look at the same sky, so their errors are correlated. The physical cross-covariance is the
file produced by [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) (`cross_cov` in
`defaults/paths.yaml`) via `CrossCov.from_cmb_lensing(roles.mflike, roles.lensing)`. It keys the block
by the components' real names and carries their auto-covariances, so it drops straight into the
likelihood through `cross_cov_path`. We rebuild with `cross_cov=True` and compare the joint
log-likelihood with and without the cross term.

In [ ]:
if not XCOV.is_file():
    print("Cross-covariance not found:", XCOV)
    print("Run create_cross_covariance.ipynb (RUN_FULL = True) first to produce it.")
else:
    model_xcov = get_model(iso_info(cross_cov=True))
    print("loglike without cross-cov:", round(loglike(model), 4))
    print("loglike with    cross-cov:", round(loglike(model_xcov), 4))

## 3. Running an MCMC

Sampling is Python-native. Passing `sample=[...]` to `iso_info` keeps the named parameters sampled
(with their priors from `defaults/`); the `sampler` block already comes from
`defaults/sampler.yaml`, so the `info` goes straight to `cobaya.run`. The cell below sets up a short
demo chain over `tau`, guarded by `RUN_MCMC` so the notebook runs top-to-bottom without launching a
multi-minute sampler.

In [ ]:
RUN_MCMC = False  # set True to launch the sampler (minutes)

info_mcmc = iso_info(sample=["tau"], cross_cov=XCOV.is_file())
# Short demo chain, on top of the settings of defaults/sampler.yaml.
info_mcmc["sampler"]["mcmc"].update({"max_samples": 50, "Rminus1_stop": 0.1})

sampled = [
    p for p, v in info_mcmc["params"].items() if isinstance(v, dict) and "prior" in v
]
print("sampled parameters:", sampled)

if RUN_MCMC:
    from cobaya import run

    info_mcmc["output"] = "chains/multigaussian"
    updated_info, sampler = run(info_mcmc)
    print("done:", sampler.products()["sample"].shape)
else:
    print("RUN_MCMC is False - skipping the sampler.")

### A YAML twin for `cobaya-run`

The Python run above can be serialized to YAML and launched from the command line. We write the joint MCMC `info` to `examples/ISO_sims.yaml`, templating the machine-specific roots as `<REPO>` (your SOLikeT checkout) and `<PACKAGES>` (your cobaya packages path) so the file is committable. Fill them in, regenerate the `sims/` twins, and run:

```bash
cobaya-run examples/ISO_sims.yaml
```

In [ ]:
from pathlib import Path

import yaml

EXAMPLES = Path("../../examples")
EXAMPLES.mkdir(exist_ok=True)
REPO = Path("../..").resolve()
PACKAGES = str(resolve_packages_path())

# YAML twin of the Python MCMC run above. Machine-specific roots are templated as
# <REPO> (this checkout) and <PACKAGES> (the cobaya packages path) so the file is
# portable and committable.
info_yaml = iso_info(sample=["tau"], cross_cov=XCOV.is_file())
info_yaml["sampler"]["mcmc"].update({"max_samples": 50, "Rminus1_stop": 0.1})
info_yaml["output"] = "chains/ISO_sims"

header = (
    "# ISO_sims joint analysis -- YAML twin of analyse_datasets.ipynb.\n"
    "# Replace <REPO> with your SOLikeT checkout and <PACKAGES> with your cobaya\n"
    "# packages path, regenerate the sims/ twins (create_datasets.ipynb +\n"
    "# create_cross_covariance.ipynb), then run:  cobaya-run examples/ISO_sims.yaml\n\n"
)
text = yaml.safe_dump(info_yaml, sort_keys=False, default_flow_style=False)
text = text.replace(str(REPO), "<REPO>").replace(PACKAGES, "<PACKAGES>")

dest = EXAMPLES / "ISO_sims.yaml"
dest.write_text(header + text)
print("wrote", dest.resolve())

## Recap

- `iso_info()` gives the joint MFLike + CMB-lensing `info` at the ISO fiducial, built from
  `defaults/` by `iso_config.build_info` and wired to fit the smooth twins in `sims/`;
  `resolve_aliases(model)` recovers `.mflike` / `.lensing`.
- The cross-covariance from [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) folds in
  through `cross_cov_path`.
- Sampling runs by calling `cobaya.run(info)` (the sampler comes from `defaults/sampler.yaml`);
  the same config is written to `examples/ISO_sims.yaml` (templated paths) for `cobaya-run`.

This closes the pipeline: [`create_datasets.ipynb`](create_datasets.ipynb) builds the data,
[`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) the cross-covariance, and this
notebook the joint fit, all at one ISO fiducial, all from `defaults/`. The full set of runs is
generated by [`generate_grid.ipynb`](generate_grid.ipynb).